# Fabric Defect Classification — ResNet18 (Main Model)

**Notebook 04** of the fabric defect classification pipeline.

Notebook 03 trained a small network from scratch and reached **0.576 macro F1**. It learned the common classes well but struggled badly with several defect types, and its held-out score stopped improving after about five epochs — a sign it had reached the limit of what 2,737 images can teach a network starting from nothing.

This notebook takes a different approach: **transfer learning**.

### What transfer learning means

ResNet18 has already been trained on ImageNet, a collection of over a million photographs. In learning to tell apart a thousand everyday objects, it built up general visual skills — detecting edges, textures, repeating patterns, breaks in a surface. Those skills are not specific to cats or cars; they apply to fabric too.

So instead of starting from random numbers, we start from that trained network and adapt it to our nine classes. The model no longer has to learn what an edge looks like from 27 examples of "Vertical" — it already knows, and only has to learn which patterns mean which defect.

### What this notebook does

1. Compares **three different ways** of adapting ResNet18, and records the results
2. Picks the best one based on validation performance
3. Trains that version across all five folds
4. Saves the trained models so they can be reused without retraining
5. Compares the result against the baseline CNN from Notebook 03

Everything else stays identical to Notebook 03 — the same folds, the same augmentation, the same metrics — so the comparison is fair and any difference comes from the model rather than the setup.

## 1. Setup

**What:** Load the libraries, find the cleaned data, and set the configuration.

**Why:** Using the same folds and the same random seed as Notebook 03 is what makes the comparison meaningful. If the two models were tested on different splits, any difference between them could just be luck.

In [ ]:
import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                             recall_score)
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

# ---------------- Configuration ----------------
MODEL_NAME = "resnet18"
IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42
NUM_WORKERS = 2
N_FOLDS = 5

EXP_EPOCHS = 15      # epochs while comparing approaches
FINAL_EPOCHS = 25    # epochs for the chosen approach (matches Notebook 03)

# Baseline CNN results from Notebook 03, used for the final comparison.
# Overwritten automatically if that notebook's summary file is found.
BASELINE = {"macro_f1": 0.576, "accuracy": 0.827}
# -----------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Find the cleaned data from Notebook 02
CANDIDATES = [Path("/content/data/processed"), Path("/kaggle/working/processed")]
kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    CANDIDATES += sorted(kaggle_input.glob("*/processed"))
    CANDIDATES += sorted(kaggle_input.glob("*"))
CANDIDATES.append(Path("data/processed"))
DATA_DIR = next((c for c in CANDIDATES if (c / "manifest.csv").exists()), None)

# Output locations
if Path("/kaggle/working").exists():
    ROOT = Path("/kaggle/working")
elif Path("/content").exists():
    ROOT = Path("/content")
else:
    ROOT = Path(".")

RESULTS_DIR = ROOT / "outputs" / MODEL_NAME
MODELS_DIR = ROOT / "models"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()
                      else "cpu")

print("Device  :", device)
print("Results :", RESULTS_DIR)
print("Models  :", MODELS_DIR)
print("Data    :", DATA_DIR if DATA_DIR else "not found - the next cell rebuilds it")

### If the cleaned data is missing

Seeing `Data : not found` is normal rather than an error. Colab and Kaggle give every notebook its own machine, so this one cannot see the files Notebook 02 wrote.

The cell below rebuilds the dataset from the raw images, repeating exactly what Notebook 02 did — the same duplicate removal, the same 224×224 conversion, the same random seed. The folds come out identical, so results stay comparable. It takes about five minutes.

In [ ]:
if DATA_DIR is None:
    import hashlib
    import os
    import shutil
    from collections import Counter

    from sklearn.model_selection import StratifiedKFold

    print("Rebuilding the cleaned dataset from the raw images.\n")

    ds2 = "/kaggle/input/multi-class-fabric-defect-detection-dataset/Dataset"
    if not os.path.exists(ds2):
        import kagglehub
        ds2 = os.path.join(
            kagglehub.dataset_download("ziya07/multi-class-fabric-defect-detection-dataset"),
            "Dataset")
    raw_classes = sorted(os.listdir(ds2))

    def dhash(path, size=16):
        with Image.open(path) as img:
            small = img.convert("L").resize((size + 1, size), Image.BILINEAR)
        px = list(small.getdata())
        bits = []
        for row in range(size):
            for col in range(size):
                bits.append("1" if px[row * (size + 1) + col] > px[row * (size + 1) + col + 1] else "0")
        return "".join(bits)

    candidates = [(c, p) for c in raw_classes
                  for p in sorted((Path(ds2) / c).glob("*")) if "_processed" not in p.name]
    print(f"After removing _processed     : {len(candidates)}")

    seen, after_exact = set(), []
    for c, p in candidates:
        h = hashlib.md5(p.read_bytes()).hexdigest()
        if h not in seen:
            seen.add(h)
            after_exact.append((c, p))
    print(f"After removing exact copies   : {len(after_exact)}")

    seen2, keep = set(), []
    for c, p in after_exact:
        k = (c, dhash(p))
        if k not in seen2:
            seen2.add(k)
            keep.append((c, p))
    print(f"After removing near-duplicates: {len(keep)}")

    labels = [c for c, _ in keep]
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_of = {}
    for f, (_, te) in enumerate(skf.split(range(len(keep)), labels)):
        for i in te:
            fold_of[i] = f

    OUT = ROOT / "data" / "processed"
    if OUT.exists():
        shutil.rmtree(OUT)
    (OUT / "images").mkdir(parents=True)

    rows = []
    for i, (c, p) in enumerate(keep):
        d = OUT / "images" / c
        d.mkdir(exist_ok=True)
        name = f"{i:05d}.jpg"
        with Image.open(p) as im:
            im.convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR).save(d / name, quality=95)
        rows.append({"filepath": str(Path("images") / c / name), "class": c,
                     "fold": fold_of[i], "source_file": p.name})
    pd.DataFrame(rows).to_csv(OUT / "manifest.csv", index=False)

    counts = Counter(labels)
    weights = {c: len(labels) / (len(raw_classes) * counts[c]) for c in raw_classes}
    json.dump(weights, open(OUT / "class_weights.json", "w"), indent=2)

    DATA_DIR = OUT
    print("\nRebuilt at:", DATA_DIR)
else:
    print("Using the cleaned data at:", DATA_DIR)

In [ ]:
manifest = pd.read_csv(DATA_DIR / "manifest.csv")
class_weights_raw = json.load(open(DATA_DIR / "class_weights.json"))
class_names = sorted(manifest["class"].unique())
n_classes = len(class_names)
cls_to_idx = {c: i for i, c in enumerate(class_names)}

# Pick up the baseline CNN results if Notebook 03 ran in this session
for p in (ROOT / "outputs" / "baseline_cnn" / "baseline_cnn_summary.json",
          Path("outputs/baseline_cnn/baseline_cnn_summary.json")):
    if p.exists():
        s = json.load(open(p))
        BASELINE = {"macro_f1": s["macro_f1"], "accuracy": s["accuracy"]}
        print("Loaded baseline results from", p)
        break
else:
    print("Using the baseline figures recorded in the configuration above.")

print(f"\nImages : {len(manifest)}  |  Classes: {n_classes}  |  Folds: {manifest['fold'].nunique()}")
print(f"Baseline CNN to beat: macro F1 {BASELINE['macro_f1']:.3f}, accuracy {BASELINE['accuracy']:.3f}")

**Observations:** _(fill in — did the data load or rebuild, and were the baseline figures found?)_

## 2. The Data Pipeline

**What:** Set up image loading and augmentation, exactly as in Notebook 03.

**Why:** For the comparison to be fair, only *one* thing may change between the two notebooks: the model. If ResNet18 also received different augmentation or different normalisation, we could not tell which change caused any improvement.

Two details are worth restating. **No 90° rotations** — turning an image a quarter turn would change a "Vertical" defect into a "horizontal" one while the label stayed the same, creating deliberately wrong training data. And the **normalisation values** are ImageNet's, which matters more here than it did in Notebook 03: ResNet18 was originally trained on images scaled this exact way, so feeding it anything else would waste part of what it already knows.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(10),          # small angles only - see note above
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class FabricDataset(Dataset):
    def __init__(self, df, root, transform):
        self.df = df.reset_index(drop=True)
        self.root = Path(root)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.root / row["filepath"]).convert("RGB")
        return self.transform(img), cls_to_idx[row["class"]]


def make_loader(df, transform, shuffle):
    return DataLoader(FabricDataset(df, DATA_DIR, transform), batch_size=BATCH_SIZE,
                      shuffle=shuffle, num_workers=NUM_WORKERS)


print("Pipeline ready - identical to Notebook 03.")

## 3. Building ResNet18

**What:** Load ResNet18 with its ImageNet training already in place, and replace its final layer.

**Why replace the last layer?** The original network ends with a layer that produces 1,000 numbers, one per ImageNet category. We need 9, one per fabric class, so that final layer is swapped for a new one. Everything before it — the part that actually recognises edges, textures and patterns — is kept.

**Two ways to train it, and we will test both:**

- **Frozen backbone.** Lock every layer except the new final one. The network becomes a fixed feature extractor and we only learn how to combine what it already sees. Fast, and with only 2,737 images there is little to overfit, but the features stay generic and were never tuned for fabric.
- **Full fine-tuning.** Let every layer keep learning, usually at a small learning rate so the existing knowledge is refined rather than destroyed. Slower and riskier on a small dataset, but the features can adapt to fabric texture specifically.

There is no way to know in advance which wins on this dataset, which is exactly why §5 measures it instead of guessing.

In [ ]:
def build_resnet18(n_classes, freeze_backbone=False):
    # Load ResNet18 with its ImageNet training already in place.
    # The `weights=` form needs torchvision 0.13+; older versions use `pretrained=`.
    try:
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    except AttributeError:
        model = models.resnet18(pretrained=True)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    # New final layer for our 9 classes (always trainable)
    model.fc = nn.Linear(model.fc.in_features, n_classes)
    return model


for frozen in (True, False):
    m = build_resnet18(n_classes, freeze_backbone=frozen)
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    label = "frozen backbone" if frozen else "full fine-tuning"
    print(f"{label:18s} total {total:,}  trainable {trainable:,}  ({trainable/total:.1%})")

print(f"\nBaseline CNN from Notebook 03 had 391,689 parameters in total.")

**Observations:** _(fill in — how many parameters does ResNet18 have compared with the baseline CNN, and how many are actually trained in each setting?)_

## 4. The Training Routine

**What:** One function that trains a model and reports how it does on held-out images.

**Why the same class weights again?** Notebook 02 calculated a weight for every class based on how rare it is, and Notebook 03 used them. Getting a "Vertical" image wrong costs about 62 times as much as getting a "defect free" image wrong. Keeping this identical means the two models are being asked to solve the same problem.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss, preds, trues = 0.0, [], []
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            preds.append(out.argmax(1).cpu())
            trues.append(y.cpu())

    return (total_loss / len(loader.dataset),
            torch.cat(preds).numpy(),
            torch.cat(trues).numpy())


def train_model(train_df, eval_df, freeze, lr, epochs, seed=SEED, verbose=True):
    torch.manual_seed(seed)
    model = build_resnet18(n_classes, freeze_backbone=freeze).to(device)

    weights = torch.tensor([class_weights_raw[c] for c in class_names],
                           dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=lr)

    train_loader = make_loader(train_df, train_tf, shuffle=True)
    eval_loader = make_loader(eval_df, eval_tf, shuffle=False)

    history = []
    for epoch in range(1, epochs + 1):
        tr_loss, _, _ = run_epoch(model, train_loader, criterion, optimizer)
        ev_loss, ev_pred, ev_true = run_epoch(model, eval_loader, criterion)
        ev_f1 = f1_score(ev_true, ev_pred, average="macro", zero_division=0)
        history.append({"epoch": epoch, "train_loss": tr_loss,
                        "eval_loss": ev_loss, "eval_f1": ev_f1})
        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f"    epoch {epoch:2d}/{epochs}  train loss {tr_loss:.3f}  "
                  f"| held-out loss {ev_loss:.3f} f1 {ev_f1:.3f}")

    return model, pd.DataFrame(history), ev_pred, ev_true


print("Training routine defined.")

## 5. Comparing Three Approaches

**What:** Train three versions of ResNet18 and compare them on a validation set.

**Why a separate validation set?** We need to choose between approaches, and that choice has to be made using data that is *not* used for the final score. Otherwise we would be picking whichever setup happened to look best on the test data, and the reported result would be flattering rather than honest.

So the folds are used like this:

| Folds | Role |
|---|---|
| 0, 1, 2 | Training |
| 3 | Validation — used to choose the approach |
| 4 | Untouched during this step |

**The three approaches:**

| Name | Setup | Question it answers |
|---|---|---|
| `frozen_backbone` | Only the new final layer learns, lr 0.001 | Are ImageNet's ready-made features enough on their own? |
| `finetune_lr1e-4` | Every layer learns, small lr 0.0001 | Does gently adapting the features to fabric help? |
| `finetune_lr1e-3` | Every layer learns, larger lr 0.001 | Does a larger step damage what the network already knows? |

The third exists to test a real risk: fine-tuning a pretrained network with too large a learning rate can overwrite its knowledge in the first few batches, leaving it no better than training from scratch.

Every run is appended to an experiment log so the whole comparison is recorded rather than remembered.

In [ ]:
EXPERIMENTS = [
    {"name": "frozen_backbone", "freeze": True,  "lr": 1e-3},
    {"name": "finetune_lr1e-4", "freeze": False, "lr": 1e-4},
    {"name": "finetune_lr1e-3", "freeze": False, "lr": 1e-3},
]

train_df = manifest[manifest["fold"].isin([0, 1, 2])]
val_df = manifest[manifest["fold"] == 3]
print(f"Train: {len(train_df)} images   Validation: {len(val_df)} images\n")

exp_log, exp_histories = [], {}
start_all = time.time()

for cfg in EXPERIMENTS:
    print(f"--- {cfg['name']} (freeze={cfg['freeze']}, lr={cfg['lr']}) ---")
    t0 = time.time()
    _, hist, pred, true = train_model(
        train_df, val_df, cfg["freeze"], cfg["lr"], EXP_EPOCHS)
    mins = (time.time() - t0) / 60

    val_f1 = f1_score(true, pred, average="macro", zero_division=0)
    val_acc = (pred == true).mean()
    exp_log.append({**cfg, "epochs": EXP_EPOCHS, "val_macro_f1": val_f1,
                    "val_accuracy": val_acc, "minutes": round(mins, 1)})
    exp_histories[cfg["name"]] = hist
    print(f"  -> validation macro F1 {val_f1:.3f} | accuracy {val_acc:.3f} "
          f"| {mins:.1f} min\n")

exp_df = pd.DataFrame(exp_log).sort_values("val_macro_f1", ascending=False)
exp_df.to_csv(RESULTS_DIR / f"{MODEL_NAME}_experiments.csv", index=False)

print(f"Total: {(time.time() - start_all)/60:.1f} minutes\n")
print(exp_df.to_string(index=False))

**Observations:** _(fill in — which approach won, by how much, and did the larger learning rate damage the pretrained network as expected?)_

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, hist in exp_histories.items():
    axes[0].plot(hist["epoch"], hist["train_loss"], label=name)
    axes[1].plot(hist["epoch"], hist["eval_f1"], label=name)

axes[0].set_title("Training loss"); axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss")
axes[1].set_title("Validation macro F1"); axes[1].set_xlabel("epoch"); axes[1].set_ylabel("macro F1")
for ax in axes:
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / f"{MODEL_NAME}_experiments.png", dpi=150, bbox_inches="tight")
plt.show()

**Observations:** _(fill in — how do the three curves differ? Does any approach improve quickly then stall, or keep climbing to the end?)_

## 6. Training the Chosen Approach on All Five Folds

**What:** Take the approach that scored best on validation and train it five times, holding out a different fold each time.

**Why:** This is the same procedure Notebook 03 used, so the two results can be compared directly. Every image ends up with a prediction from a model that never saw it during training, which matters most for "Vertical" — all 27 of its images get tested rather than roughly 4.

**One caveat, stated openly.** The approach was chosen using fold 3, and fold 3 also appears in this cross-validation. That makes the final number very slightly optimistic. Fold 4 was never involved in the choice, so its score is a clean check: if it broadly matches the others, the selection did not distort the result.

Each fold's trained weights are saved, so predictions can be made later without repeating any of this.

In [ ]:
best = exp_df.iloc[0]
BEST_CFG = {"freeze": bool(best["freeze"]), "lr": float(best["lr"])}
print(f"Chosen approach: {best['name']}  (freeze={BEST_CFG['freeze']}, lr={BEST_CFG['lr']})")
print(f"Validation macro F1 was {best['val_macro_f1']:.3f}\n")

oof_pred = np.zeros(len(manifest), dtype=int)
oof_true = np.zeros(len(manifest), dtype=int)
fold_scores, histories = [], []

start = time.time()
for fold in sorted(manifest["fold"].unique()):
    print(f"Fold {fold}")
    tr = manifest[manifest["fold"] != fold]
    te = manifest[manifest["fold"] == fold]

    model, hist, pred, true = train_model(
        tr, te, BEST_CFG["freeze"], BEST_CFG["lr"], FINAL_EPOCHS, seed=SEED + fold)

    idx = te.index.values
    oof_pred[idx] = pred
    oof_true[idx] = true
    hist["fold"] = fold
    histories.append(hist)

    f1 = f1_score(true, pred, average="macro", zero_division=0)
    acc = (pred == true).mean()
    fold_scores.append({"fold": fold, "macro_f1": f1, "accuracy": acc})

    torch.save(model.state_dict(), MODELS_DIR / f"{MODEL_NAME}_fold{fold}.pt")
    print(f"  -> macro F1 {f1:.3f} | accuracy {acc:.3f} | weights saved\n")

history_df = pd.concat(histories, ignore_index=True)
scores_df = pd.DataFrame(fold_scores)

print(f"Total training time: {(time.time() - start)/60:.1f} minutes\n")
print(scores_df.to_string(index=False))
print(f"\nMean macro F1: {scores_df['macro_f1'].mean():.3f} "
      f"(+/- {scores_df['macro_f1'].std():.3f})")
print(f"Fold 4 (never used for choosing the approach): "
      f"{scores_df[scores_df['fold'] == 4]['macro_f1'].iloc[0]:.3f}")

**Observations:** _(fill in — how much do the folds vary compared with the baseline CNN's spread of ±0.075? Is fold 4 in line with the rest?)_

## 7. Results

**What:** Score every prediction collected across the five folds.

**Why:** Each image was predicted by a model that had never seen it, so these numbers describe genuinely unseen data. Macro F1 is the headline because it counts all nine classes equally; per-class recall shows whether the rare defects are actually being found.

In [ ]:
macro_f1 = f1_score(oof_true, oof_pred, average="macro", zero_division=0)
accuracy = (oof_pred == oof_true).mean()

print(f"Overall macro F1 : {macro_f1:.3f}")
print(f"Overall accuracy : {accuracy:.3f}\n")
print(classification_report(oof_true, oof_pred, target_names=class_names,
                            digits=3, zero_division=0))

**Observations:** _(fill in — which classes improved most over the baseline CNN, and which are still weak?)_

In [ ]:
recalls = recall_score(oof_true, oof_pred, average=None, zero_division=0)
counts = manifest["class"].value_counts()

# Baseline CNN per-class recall from Notebook 03, for side-by-side comparison
baseline_recall = {
    "Vertical": 0.222, "horizontal": 0.588, "Pinched fabric": 0.833,
    "Needle mark": 0.954, "Broken stitch": 0.250, "hole": 0.220,
    "lines": 0.712, "stain": 0.809, "defect free": 0.937,
}

compare = pd.DataFrame({
    "class": class_names,
    "images": [counts[c] for c in class_names],
    "baseline_recall": [baseline_recall.get(c, float("nan")) for c in class_names],
    "resnet18_recall": recalls,
}).sort_values("images")
compare["change"] = compare["resnet18_recall"] - compare["baseline_recall"]

print("Recall by class, rarest first:")
print(compare.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

**Observations:** _(fill in — did the rare classes improve? Did any class get worse?)_

In [ ]:
cm = confusion_matrix(oof_true, oof_pred)
cm_pct = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(n_classes)); ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticks(range(n_classes)); ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("ResNet18 — confusion matrix (row %)")

for i in range(n_classes):
    for j in range(n_classes):
        if cm[i, j] > 0:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                    color="white" if cm_pct[i, j] > 0.5 else "black")

plt.colorbar(im, label="fraction of actual class")
plt.tight_layout()
plt.savefig(RESULTS_DIR / f"{MODEL_NAME}_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

**Observations:** _(fill in — are the same confusions still present? Notebook 03 collapsed Broken stitch into Needle mark 79 times out of 112, called Vertical "horizontal" 18 times out of 27, and passed 75 stains as defect free.)_

## 8. Did Transfer Learning Help?

**What:** Put the two models side by side.

**Why this is the point of the project.** Both were trained on the same images, the same folds, the same augmentation and the same class weights. The only difference is that ResNet18 started from a network already trained on a million photographs. Any gap between them is therefore attributable to that pretraining — which is the claim this project set out to test.

The safety-critical number is also included: how many defective rolls would be passed as clean fabric.

In [ ]:
missed_defects = int(sum(cm[i, class_names.index("defect free")]
                         for i, c in enumerate(class_names) if c != "defect free"))
total_defects = int(sum(counts[c] for c in class_names if c != "defect free"))

comparison = pd.DataFrame([
    {"model": "Baseline CNN (from scratch)", "macro_f1": BASELINE["macro_f1"],
     "accuracy": BASELINE["accuracy"]},
    {"model": "ResNet18 (transfer learning)", "macro_f1": macro_f1, "accuracy": accuracy},
])
comparison["macro_f1_gain"] = comparison["macro_f1"] - BASELINE["macro_f1"]

print(comparison.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print(f"\nDefects wrongly passed as 'defect free': {missed_defects} of {total_defects} "
      f"({missed_defects/total_defects:.1%})")
print(f"Baseline CNN missed 78 of 1,074 (7.3%) for comparison.")

**Observations:** _(fill in — how much did macro F1 improve, and did the number of missed defects fall?)_

## 9. Save the Model and Its Settings

**What:** Write out the results, and save the settings needed to reload the model later.

**Why the settings matter as much as the weights.** A saved model is a list of numbers; on its own it cannot be used. Anything making predictions later has to know which class each output position means, what size images to expect, and which normalisation values were applied. Saving those alongside the weights is what makes the model reusable — a prediction notebook can then load it in a few lines without repeating any training.

In [ ]:
scores_df.to_csv(RESULTS_DIR / f"{MODEL_NAME}_fold_scores.csv", index=False)
history_df.to_csv(RESULTS_DIR / f"{MODEL_NAME}_history.csv", index=False)

pred_df = manifest.copy()
pred_df["predicted"] = [class_names[i] for i in oof_pred]
pred_df["correct"] = pred_df["predicted"] == pred_df["class"]
pred_df.to_csv(RESULTS_DIR / f"{MODEL_NAME}_predictions.csv", index=False)

summary = {
    "model": "ResNet18 (transfer learning)",
    "chosen_approach": best["name"],
    "freeze_backbone": BEST_CFG["freeze"],
    "learning_rate": BEST_CFG["lr"],
    "epochs": FINAL_EPOCHS,
    "n_images": int(len(manifest)),
    "n_classes": n_classes,
    "n_folds": int(manifest["fold"].nunique()),
    "macro_f1": float(macro_f1),
    "accuracy": float(accuracy),
    "per_class_recall": {c: float(r) for c, r in zip(class_names, recalls)},
    "baseline_cnn_macro_f1": BASELINE["macro_f1"],
    "missed_defects": missed_defects,
    "total_defects": total_defects,
}
json.dump(summary, open(RESULTS_DIR / f"{MODEL_NAME}_summary.json", "w"), indent=2)

# Everything a prediction script needs in order to reload the model
model_config = {
    "architecture": "resnet18",
    "class_names": class_names,
    "img_size": IMG_SIZE,
    "normalize_mean": IMAGENET_MEAN,
    "normalize_std": IMAGENET_STD,
    "freeze_backbone": BEST_CFG["freeze"],
    "weight_files": [f"{MODEL_NAME}_fold{f}.pt" for f in sorted(manifest["fold"].unique())],
}
json.dump(model_config, open(MODELS_DIR / f"{MODEL_NAME}_config.json", "w"), indent=2)

# Copy the figure into assets/ so it can be shown in the README
ASSETS = Path("assets")
if ASSETS.exists():
    import shutil
    shutil.copy2(RESULTS_DIR / f"{MODEL_NAME}_confusion_matrix.png", ASSETS)
    print("Confusion matrix copied to assets/\n")

print("Results:", RESULTS_DIR)
for f in sorted(RESULTS_DIR.iterdir()):
    print(f"   {f.name:40s} {f.stat().st_size/1024:8.1f} KB")
print("\nModels:", MODELS_DIR)
for f in sorted(MODELS_DIR.iterdir()):
    print(f"   {f.name:40s} {f.stat().st_size/1024:8.1f} KB")

## 10. Summary

### What this notebook did

1. Loaded the same 2,737 images and the same five folds used by Notebook 03.
2. Compared **three ways** of adapting ResNet18 — frozen backbone, gentle fine-tuning, and aggressive fine-tuning — using a validation fold that played no part in the final score.
3. Trained the winning approach across all five folds, so every image was tested by a model that had never seen it.
4. **Saved the trained weights and the settings needed to reload them**, so predictions can be made later without retraining.
5. Compared the result directly against the baseline CNN, changing nothing except the model.

### Files produced

```
outputs/resnet18/
├── resnet18_summary.json          headline scores and per-class recall
├── resnet18_experiments.csv       the three approaches and their scores
├── resnet18_experiments.png       training curves for the comparison
├── resnet18_fold_scores.csv       score for each fold
├── resnet18_history.csv           loss and F1 after every epoch
├── resnet18_predictions.csv       the prediction for every image
└── resnet18_confusion_matrix.png  confusion matrix figure

models/
├── resnet18_fold0.pt … fold4.pt   trained weights, one per fold
└── resnet18_config.json           class names, image size, normalisation
```

### Decisions worth defending

- **Three approaches compared rather than one tried.** Whether to freeze the pretrained layers is a real choice with arguments on both sides, so it was measured rather than assumed.
- **A separate validation fold for choosing.** Selecting the approach on the test data would have inflated the final score. Fold 3 made the choice; fold 4 stayed out of it entirely and acts as a clean check.
- **Everything except the model held constant.** Same folds, same augmentation, same class weights, same metrics, same seed — so the comparison isolates the effect of pretraining.
- **A deliberately bad configuration included.** Fine-tuning at lr 0.001 was expected to damage the pretrained features. Testing it demonstrates *why* the chosen learning rate is right, rather than merely asserting it.
- **Weights and settings saved together.** Weights alone cannot be used; the class order, image size and normalisation values are what make them reloadable.

### Honest limitations

- The approach was chosen using fold 3, which also appears in the final cross-validation, so the headline figure is slightly optimistic. Fold 4's score is reported separately as a clean comparison.
- "Vertical" (27 images) and "horizontal" (34) remain very small. Their per-class scores carry real uncertainty no matter which model is used.
- The dataset combines several source collections, so some source-related bias may remain, as recorded in Notebook 01.

### Next: prediction on new images

With the weights and settings saved, a prediction notebook can load the model and classify a fabric image it has never seen — the end-to-end demo the project set out to build.